# CogniFlow - Stage 1: Semantic Task Analysis (LLaMA-3)
Loads O\*NET task descriptions, runs them through LLaMA-3.1-8B-Instruct,
parses structured outputs, and saves `task_dataset.csv` for Stage 3 fusion.

In [1]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

CUDA available: True
GPU name: Tesla T4
VRAM (GB): 15.6


In [2]:
!pip install -q transformers accelerate bitsandbytes huggingface_hub requests pandas

## HuggingFace Login

In [3]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

hf_token = UserSecretsClient().get_secret('HF_TOKEN')
login(hf_token)
print('Logged in.')

Logged in.


## Step 1 - Load O\*NET Task Statements
Downloads the official Task Statements TSV from O\*NET and samples 400 tasks
stratified across job categories so the complexity distribution is balanced.

In [4]:
import os
for f in os.listdir('/kaggle/input/datasets/tejasgovind/onet-task-statements/'):
    print(f)

Task Statements.xlsx


In [5]:
import pandas as pd
import random

onet = pd.read_excel('/kaggle/input/datasets/tejasgovind/onet-task-statements/Task Statements.xlsx')
print('Columns:', onet.columns.tolist())
print('Total rows:', len(onet))

Columns: ['O*NET-SOC Code', 'Title', 'Task ID', 'Task', 'Task Type', 'Incumbents Responding', 'Date', 'Domain Source']
Total rows: 19530


In [6]:
# Stratified sample: pick tasks evenly across O*NET-SOC codes
# so we don't over-sample one job family
random.seed(42)

onet_clean = onet[['O*NET-SOC Code', 'Task']].dropna().drop_duplicates(subset='Task')

# Sample up to 2 tasks per SOC code, then take 400 total
grouped = (
    onet_clean
    .groupby('O*NET-SOC Code')['Task']
    .apply(lambda x: x.sample(min(len(x), 2), random_state=42))
    .reset_index(drop=True)
)

tasks = grouped.sample(min(400, len(grouped)), random_state=42).tolist()

print(f'Sampled {len(tasks)} tasks from {onet_clean["O*NET-SOC Code"].nunique()} SOC codes')
print('\nExamples:')
for t in tasks[:5]:
    print(' -', t)

Sampled 400 tasks from 974 SOC codes

Examples:
 - Transfer tools, parts, equipment, and supplies to and from work stations and other areas.
 - Inspect floors for smoothness.
 - Fit and study garments on customers to determine required alterations.
 - Discuss traffic routing plans and control point locations with superiors.
 - Prepare final project layout drawings that include details such as stress calculations.


## Step 2 - Load LLaMA-3.1-8B-Instruct (4-bit)

In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = 'meta-llama/Meta-Llama-3.1-8B-Instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map='auto',
    quantization_config=bnb_config
)
print('Model loaded.')

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Model loaded.


## Step 3 - Define System Prompt & Parser

In [8]:
SYSTEM_PROMPT = """You are a workplace cognitive load analyzer for employees with ADHD.
Given a professional task description, respond ONLY in this exact format with no extra text:

COMPLEXITY: <integer 1-5>
STEPS: <integer>
PRIORITY: <low|medium|high>
RECO: <one actionable sentence, max 12 words>

Scoring rules:
- COMPLEXITY 1 = trivial (e.g. send a single email)
- COMPLEXITY 2 = simple, one clear action (e.g. schedule a meeting)
- COMPLEXITY 3 = moderate, a few coordinated steps (e.g. write a status report)
- COMPLEXITY 4 = complex, multiple dependencies or stakeholders
- COMPLEXITY 5 = highly demanding (e.g. architect a system, lead cross-team delivery)
- STEPS = realistic number of discrete actions to complete this task
- PRIORITY = urgency relative to a standard workday
- RECO = one concrete first step or coping strategy for someone with ADHD
Output ONLY the four lines above. No preamble, no explanation."""

In [ ]:
import re

PARSE_FAILURES = []  

def parse_output(raw: str):
    try:
        fields = {}
        for line in raw.strip().splitlines():
            line = line.strip()
            if ':' in line:
                key, _, value = line.partition(':')
                fields[key.strip().upper()] = value.strip()

        complexity = max(1, min(5, int(fields['COMPLEXITY'])))
        steps = max(1, int(fields['STEPS']))
        priority = fields['PRIORITY'].lower().strip()
        if priority not in ('low', 'medium', 'high'):
            raise ValueError(f'Bad priority: {priority}')
        reco = fields['RECO']

        return {
            'complexity_score': complexity,
            'estimated_steps': steps,
            'priority_level': priority,
            'recommendation': reco
        }
    except Exception as e:
        PARSE_FAILURES.append({'error': str(e), 'raw': raw[:200]})
        return None

print('parse_output defined.')

parse_output defined.


In [ ]:
def analyze_task(task_description: str):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': f'Task: {task_description}'}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors='pt',
        return_dict=True
    ).to('cuda')

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=80,
            temperature=1.0,  
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    full_response = tokenizer.decode(output[0], skip_special_tokens=True)
    assistant_text = full_response.split('assistant')[-1].strip()
    return parse_output(assistant_text)

print('analyze_task defined.')

analyze_task defined.


## Step 4 - Smoke Test on Anchor Tasks
Verify the model produces sensible, monotonically ordered complexity scores
before running the full 400-task batch.

In [11]:
anchor_tasks = [
    ('trivial',  "Reply to John's email about the Tuesday meeting."),
    ('simple',   'Schedule a one-hour team sync for next week.'),
    ('moderate', 'Prepare a weekly KPI summary for leadership.'),
    ('complex',  'Review the pull request for the auth service and leave feedback.'),
    ('complex',  'Debug why the production database is timing out.'),
    ('demanding','Coordinate with four teams to deliver the migration plan.'),
    ('demanding','Architect a new microservices migration plan for the entire platform.')
]

print(f'{'Label':<12} {'C':>2} {'S':>5} {'Priority':<8}  Task')
print('-' * 90)
for label, task in anchor_tasks:
    r = analyze_task(task)
    if r:
        print(f"{label:<12} {r['complexity_score']:>2} {r['estimated_steps']:>5} {r['priority_level']:<8}  {task[:60]}")
    else:
        print(f'{label:<12}  PARSE FAILED  {task[:60]}')

The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Label         C     S Priority  Task
------------------------------------------------------------------------------------------
trivial       2     3 low       Reply to John's email about the Tuesday meeting.
simple        2     3 medium    Schedule a one-hour team sync for next week.
moderate      3     5 medium    Prepare a weekly KPI summary for leadership.
complex       2     3 medium    Review the pull request for the auth service and leave feedb
complex       5     7 high      Debug why the production database is timing out.
demanding     5    12 high      Coordinate with four teams to deliver the migration plan.
demanding     5    17 high      Architect a new microservices migration plan for the entire 


## Step 5 - Batch Run Over 400 O\*NET Tasks
Checkpoints to CSV every 50 tasks so progress is never lost.

In [ ]:
import time

results = []
CHECKPOINT_EVERY = 50
CHECKPOINT_PATH = '/kaggle/working/task_dataset_checkpoint.csv'

start = time.time()

for i, task in enumerate(tasks):
    result = analyze_task(task)
    if result:
        result['task_description'] = task
        results.append(result)

    if (i + 1) % CHECKPOINT_EVERY == 0 or i == len(tasks) - 1:
        elapsed = time.time() - start
        rate = (i + 1) / elapsed
        remaining = (len(tasks) - i - 1) / rate if rate > 0 else 0
        print(f'[{i+1:>3}/{len(tasks)}] parsed={len(results)} '
              f'failed={i+1-len(results)} '
              f'elapsed={elapsed/60:.1f}m eta={remaining/60:.1f}m')
        pd.DataFrame(results).to_csv(CHECKPOINT_PATH, index=False)

print(f'\nDone. {len(results)}/{len(tasks)} tasks parsed successfully.')
if PARSE_FAILURES:
    print(f'{len(PARSE_FAILURES)} parse failures. First 3:')
    for f in PARSE_FAILURES[:3]:
        print(' ', f)

[ 50/400] parsed=50 failed=0 elapsed=3.0m eta=20.7m
[100/400] parsed=100 failed=0 elapsed=6.0m eta=17.9m
[150/400] parsed=150 failed=0 elapsed=9.0m eta=15.0m
[200/400] parsed=200 failed=0 elapsed=12.1m eta=12.1m
[250/400] parsed=250 failed=0 elapsed=15.1m eta=9.0m
[300/400] parsed=300 failed=0 elapsed=18.0m eta=6.0m
[350/400] parsed=350 failed=0 elapsed=21.0m eta=3.0m
[400/400] parsed=400 failed=0 elapsed=24.0m eta=0.0m

Done. 400/400 tasks parsed successfully.


## Step 6 - Distribution Check
Verify complexity coverage before saving. Aim for at least some tasks at every level 1–5.

In [ ]:
df = pd.DataFrame(results)

print('=== Complexity distribution ===')
print(df['complexity_score'].value_counts().sort_index())

print('\n=== Priority distribution ===')
print(df['priority_level'].value_counts())

print('\n=== Steps summary ===')
print(df['estimated_steps'].describe().round(2))

for level in range(1, 6):
    count = (df['complexity_score'] == level).sum()
    flag = '⚠️  LOW' if count < 10 else '✅'
    print(f'Complexity {level}: {count} tasks  {flag}')

=== Complexity distribution ===
complexity_score
1      2
2     60
3     99
4    174
5     65
Name: count, dtype: int64

=== Priority distribution ===
priority_level
high      259
medium    122
low        19
Name: count, dtype: int64

=== Steps summary ===
count    400.00
mean       7.20
std        2.88
min        2.00
25%        7.00
50%        7.00
75%        7.00
max       20.00
Name: estimated_steps, dtype: float64
Complexity 1: 2 tasks  ⚠️  LOW
Complexity 2: 60 tasks  ✅
Complexity 3: 99 tasks  ✅
Complexity 4: 174 tasks  ✅
Complexity 5: 65 tasks  ✅


## Step 7 - Encode Features & Save Final Dataset

In [ ]:
def encode_priority(p):
    return {'low': 0, 'medium': 1, 'high': 2}.get(p, 1)

df['priority_encoded'] = df['priority_level'].apply(encode_priority)

# Final feature columns we need for Stage 3
FEATURE_COLS = ['complexity_score', 'estimated_steps', 'priority_encoded']

OUT_CSV  = '/kaggle/working/task_dataset.csv'
OUT_JSON = '/kaggle/working/task_dataset.json'

df.to_csv(OUT_CSV, index=False)
df.to_json(OUT_JSON, orient='records', indent=2)

print(f'✅ Saved {len(df)} rows to task_dataset.csv')
print(f'✅ Saved task_dataset.json')
print(f'\nColumns: {df.columns.tolist()}')
print(df[FEATURE_COLS].head(10))

✅ Saved 400 rows to task_dataset.csv
✅ Saved task_dataset.json

Columns: ['complexity_score', 'estimated_steps', 'priority_level', 'recommendation', 'task_description', 'priority_encoded']
   complexity_score  estimated_steps  priority_encoded
0                 3                7                 1
1                 2                3                 1
2                 3                7                 1
3                 3                5                 2
4                 5                7                 2
5                 5               12                 2
6                 4                7                 2
7                 4               12                 2
8                 2                3                 1
9                 5                7                 2


## Step 8 - Stage 1 Helper Functions (used by Stage 3)
These are the clean API that Stage 3 will import.

In [ ]:
FEATURE_COLUMNS = [
    'complexity_score',
    'estimated_steps',
    'priority_encoded',
    'avg_gaze',
    'avg_head_pose',
    'avg_eye_openness',
    'gaze_variance'
]

def analyze_task_features(task_description: str):
    result = analyze_task(task_description)
    if result is None:
        return None
    return {
        'complexity_score': result['complexity_score'],
        'estimated_steps': result['estimated_steps'],
        'priority_encoded': encode_priority(result['priority_level'])
    }

def build_feature_row(task_description: str, user_profile: dict):
    task_features = analyze_task_features(task_description)
    if task_features is None:
        return None
    merged = {**task_features, **user_profile}
    return pd.DataFrame([merged])[FEATURE_COLUMNS]

print('Helper functions ready.')

Helper functions ready.


## Step 9 - Stage 3 Preview (Sanity Check)
Verify the feature row shape and values look right with a mock behavioral profile.

In [16]:
mock_profiles = {
    'highly_focused':    {'avg_gaze': 0.85, 'avg_head_pose': 0.88, 'avg_eye_openness': 0.030, 'gaze_variance': 0.02},
    'easily_distracted': {'avg_gaze': 0.45, 'avg_head_pose': 0.60, 'avg_eye_openness': 0.025, 'gaze_variance': 0.18},
    'fatigued':          {'avg_gaze': 0.65, 'avg_head_pose': 0.70, 'avg_eye_openness': 0.015, 'gaze_variance': 0.09},
    'moderate':          {'avg_gaze': 0.70, 'avg_head_pose': 0.75, 'avg_eye_openness': 0.028, 'gaze_variance': 0.06}
}

hard_task = 'Coordinate with four teams to deliver the Q3 migration plan.'

print(f'Task: {hard_task}\n')
for profile_name, profile in mock_profiles.items():
    row = build_feature_row(hard_task, profile)
    if row is not None:
        print(f'Profile: {profile_name}')
        print(row.to_string(index=False))
        print()

Task: Coordinate with four teams to deliver the Q3 migration plan.

Profile: highly_focused
 complexity_score  estimated_steps  priority_encoded  avg_gaze  avg_head_pose  avg_eye_openness  gaze_variance
                5               12                 2      0.85           0.88              0.03           0.02

Profile: easily_distracted
 complexity_score  estimated_steps  priority_encoded  avg_gaze  avg_head_pose  avg_eye_openness  gaze_variance
                5               12                 2      0.45            0.6             0.025           0.18

Profile: fatigued
 complexity_score  estimated_steps  priority_encoded  avg_gaze  avg_head_pose  avg_eye_openness  gaze_variance
                5               12                 2      0.65            0.7             0.015           0.09

Profile: moderate
 complexity_score  estimated_steps  priority_encoded  avg_gaze  avg_head_pose  avg_eye_openness  gaze_variance
                5               12                 2       0.7   